In [5]:
"""
Stage 5 Themes -- 04: Number Subthemes
=======================================
Assigns a stable, sortable ID to every theme and subtheme in
classified_moment_inventory_long.csv, and writes two output files:

    numbered_classified_moment_inventory_long.csv        (all moments)
    numbered_classified_moment_inventory_means_only.csv  (cwmean + raw_level only)

ID SCHEME
---------
Zero-padded, underscore-separated: THEME_SUBTHEME, e.g. "03_07", "11_02".

Chosen over the old dot-notation ('1.10') for one reason: '1.10' round-trips
through a CSV as the float 1.1 and collides with the real subtheme '1.1'. That
exact bug cost a full remediation pass in the old pipeline (Stage_3 subtheme
fix). An underscore-joined two-digit pair cannot be parsed as a float, so the
failure mode is structurally impossible here, not just avoided by convention.

Two-digit zero-padding also makes plain string sort equal numeric sort:
'02_01' < '10_01' as strings, which it would not be with unpadded '2_1' vs
'10_1'. This matters because SparseKAN.from_taxonomy() does
    sorted(tax["subtheme_id"].unique())
and that order fixes the hidden-node order printed in architecture_summary().

Themes and subthemes are both ordered ALPHABETICALLY BY NAME. This is a
deliberate departure from the old pipeline, which hand-preserved theme IDs
1-13 from an earlier assignment. That mapping does not survive here -- the
taxonomy underneath it has changed too much (merges, splits, ~331 subthemes
against the old ~99) for the old numbers to mean the same thing. Alphabetical
is the only scheme that requires no manual map and is trivially
re-derivable by anyone reading this file.

MEANS-ONLY FILE
----------------
Filters to moment in {cwmean, raw_level} and sets column = base_factor. This
matches how the assembled data actually stores these features:
    agg_full_moments  -- stock factors carry a moment suffix (Tax_cwmean)
    agg_means, panel  -- stock factors use the bare base name (Tax)
Macro raw_level features already have column == base_factor in the long
file, so the rename is a no-op for them and only bites for stock cwmean rows.

Numbering is done ONCE on the full-moments file and the means-only file
inherits those same IDs by filtering rather than renumbering. This means
means-only subtheme IDs are non-contiguous (e.g. 07_01, 07_05, 07_09 with
gaps) but a given ID refers to the same subtheme in both feature sets --
gaps are harmless since nothing indexes by position, only by value.

KNOWN NAME COLLISIONS BETWEEN STOCK AND MACRO FACTORS
------------------------------------------------------
'skew_chg_5d' exists as base_factor for TWO unrelated quantities:
    macro : 5-day change in the CBOE SKEW index (market-wide tail-risk pricing)
    stock : 5-day change in a stock's own option-implied skew (Skew_OTM)
In agg_full_moments these never collide -- the stock version carries a moment
suffix (skew_chg_5d_cwmean, skew_chg_5d_cwstd, ...) while the macro version is
bare (skew_chg_5d). The collision only appears when building the means-only
file, which strips moment suffixes back to the bare base_factor name.

Resolution: Stage 2 (Stage_4_Assembly/01_apply_union.py, MANUAL_DROP) already
renamed the stock cwmean to 'stock_skew_chg_5d' and dropped it before the
aggregate means table was assembled, so the macro row is what actually
survives in agg_means.parquet. The stock row is therefore excluded here from
the means-only file. It remains present under its own suffixed name in the
full-moments file, where no collision exists.

KNOWN_COLLISIONS below is deliberately a named, explicit dict rather than a
general "macro wins" rule. A blanket preference could silently mask a future
collision where the correct resolution runs the other way. If a new collision
appears, the assertion below will name it and stop rather than guess.

NOT HANDLED HERE: the five binary regime indicators (vix_above_20,
vix_above_30, curve_inverted_2y10y, curve_inverted_3m10y, credit_stress).
They were excluded from moment_inventory_wide upstream and never entered
this long file, so they have no theme/subtheme and are numbered nowhere.
They are removed from the assembled data entirely in a separate step
(Stage_4_Assembly/07_drop_binaries.ipynb) rather than merely left untheme'd,
since each is a coarsening of a continuous feature already present.

CONTRACT WITH THE MODEL CODE
-----------------------------
SparseKAN.from_taxonomy() and SparseMLP.from_taxonomy() require exactly:
    column, subtheme_id, subtheme_name, theme_id, theme_name
Both output files carry these five columns under exactly these names, so
no changes are needed in sparse_kan.py / sparse_mlp.py.

Downstream loaders should still read with dtype=str on the ID columns as a
defensive habit, even though the underscore format cannot be misparsed as
a float the way the old dot format could.
"""

import pandas as pd
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
THEMES_DIR = Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes')

IN_PATH        = THEMES_DIR / 'classified_moment_inventory_long.csv'
OUT_FULL_PATH  = THEMES_DIR / 'numbered_classified_moment_inventory_long.csv'
OUT_MEANS_PATH = THEMES_DIR / 'numbered_classified_moment_inventory_means_only.csv'

MEANS_MOMENTS = {'cwmean', 'raw_level'}

# Contract columns, in the order model code expects to see first.
CONTRACT_COLS = ['column', 'subtheme_id', 'subtheme_name', 'theme_id', 'theme_name']
REFERENCE_COLS = ['base_factor', 'moment', 'level', 'cadence', 'description']

# Known base_factor name collisions between the stock and macro pipelines.
# Key: base_factor name. Value: which 'level' wins when building means-only.
# See "KNOWN NAME COLLISIONS" in the module docstring above for why this
# exists and why 'macro' is correct for this specific entry.
KNOWN_COLLISIONS = {
    'skew_chg_5d': 'macro',
}


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD AND VALIDATE INPUT
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 90)
print("STAGE 5 THEMES -- 04: NUMBER SUBTHEMES")
print("=" * 90)

df = pd.read_csv(IN_PATH)
print(f"\n  Loaded: {IN_PATH.name}")
print(f"    {len(df):,} rows")

required_input_cols = {'column', 'base_factor', 'moment', 'level', 'cadence',
                        'theme', 'subtheme', 'description'}
missing = required_input_cols - set(df.columns)
assert not missing, f"Input file missing expected columns: {missing}"

# No unassigned rows. An unassigned theme/subtheme here would either crash
# the sort below or -- worse -- silently produce a numeric-looking group
# under a NaN key. Fail loudly and name the offending columns.
n_nan_theme = df['theme'].isna().sum()
n_nan_sub = df['subtheme'].isna().sum()
assert n_nan_theme == 0, (
    f"{n_nan_theme} rows have no theme assigned -- resolve in the "
    f"classification notebook before numbering. Offending columns: "
    f"{df.loc[df['theme'].isna(), 'column'].tolist()[:10]}")
assert n_nan_sub == 0, (
    f"{n_nan_sub} rows have no subtheme assigned -- resolve before "
    f"numbering. Offending columns: "
    f"{df.loc[df['subtheme'].isna(), 'column'].tolist()[:10]}")

n_dupe_cols = df['column'].duplicated().sum()
assert n_dupe_cols == 0, (
    f"{n_dupe_cols} duplicate 'column' values in the input -- each surviving "
    f"model feature should appear exactly once. Duplicates: "
    f"{df.loc[df['column'].duplicated(keep=False), 'column'].tolist()[:10]}")

print(f"    Themes (raw):     {df['theme'].nunique()}")
print(f"    Subthemes (raw):  {df.groupby(['theme', 'subtheme']).ngroups}")
print(f"    ✓ No unassigned rows, no duplicate columns")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: BUILD THEME AND SUBTHEME ID MAPS (alphabetical, zero-padded)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("STEP 2: BUILD ID MAPS")
print("=" * 90)

themes_sorted = sorted(df['theme'].unique())
n_themes = len(themes_sorted)
assert n_themes <= 99, (
    f"{n_themes} themes exceeds 99 -- two-digit theme IDs will collide. "
    f"Widen the padding in the f-string below.")

theme_id_map = {name: f"{i + 1:02d}" for i, name in enumerate(themes_sorted)}

subtheme_id_map = {}   # (theme_name, subtheme_name) -> "TT_SS"
max_subs_in_one_theme = 0

for theme_name in themes_sorted:
    subs_in_theme = sorted(
        df.loc[df['theme'] == theme_name, 'subtheme'].unique()
    )
    max_subs_in_one_theme = max(max_subs_in_one_theme, len(subs_in_theme))
    assert len(subs_in_theme) <= 99, (
        f"Theme '{theme_name}' has {len(subs_in_theme)} subthemes, exceeding "
        f"99 -- widen the padding in the f-string below.")

    t_id = theme_id_map[theme_name]
    for j, sub_name in enumerate(subs_in_theme):
        subtheme_id_map[(theme_name, sub_name)] = f"{t_id}_{j + 1:02d}"

print(f"\n  {n_themes} themes numbered 01-{n_themes:02d}")
print(f"  Largest theme has {max_subs_in_one_theme} subthemes")
print(f"\n  Theme ID assignment (alphabetical):")
for name in themes_sorted:
    n_sub = df.loc[df['theme'] == name, 'subtheme'].nunique()
    n_feat = df.loc[df['theme'] == name, 'column'].nunique()
    print(f"    {theme_id_map[name]}  {name:<40s} "
          f"{n_sub:>3d} subthemes  {n_feat:>4d} features")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: APPLY IDS TO THE FULL-MOMENTS FILE
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("STEP 3: APPLY IDS")
print("=" * 90)

df['theme_id'] = df['theme'].map(theme_id_map)
df['subtheme_id'] = [
    subtheme_id_map[(t, s)] for t, s in zip(df['theme'], df['subtheme'])
]
df = df.rename(columns={'theme': 'theme_name', 'subtheme': 'subtheme_name'})

# ── Validate: every subtheme_id maps to exactly one (theme_id, subtheme_name) ──
# Guards against a same-named subtheme appearing under two different themes,
# which the construction above cannot produce but which would silently
# corrupt from_taxonomy()'s layer-1 mask if it ever happened.
check = df.groupby('subtheme_id')[['theme_id', 'subtheme_name']].nunique()
bad = check[(check['theme_id'] > 1) | (check['subtheme_name'] > 1)]
assert len(bad) == 0, (
    f"{len(bad)} subtheme_id values map to more than one theme_id or "
    f"subtheme_name -- numbering is not well-defined:\n{bad}")

n_theme_ids = df['theme_id'].nunique()
n_subtheme_ids = df['subtheme_id'].nunique()
assert n_theme_ids == n_themes
assert n_subtheme_ids == df.groupby(['theme_id', 'subtheme_name']).ngroups

print(f"\n  ✓ {n_theme_ids} theme_id values, {n_subtheme_ids} subtheme_id "
      f"values, all well-defined")

sample = df.drop_duplicates('subtheme_id')[
    ['theme_id', 'theme_name', 'subtheme_id', 'subtheme_name']
].sort_values('subtheme_id').head(8)
print(f"\n  Sample IDs:")
print(sample.to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: REORDER COLUMNS AND SAVE FULL-MOMENTS FILE
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("STEP 4: SAVE FULL-MOMENTS FILE")
print("=" * 90)

other_cols = [c for c in df.columns if c not in CONTRACT_COLS + REFERENCE_COLS]
final_order = CONTRACT_COLS + REFERENCE_COLS + other_cols
df_full = df[final_order].copy()

df_full.to_csv(OUT_FULL_PATH, index=False)
print(f"\n  Saved: {OUT_FULL_PATH.name}")
print(f"    {len(df_full):,} rows, {len(df_full.columns)} columns")
print(f"    Columns: {list(df_full.columns)}")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: BUILD MEANS-ONLY FILE (filter, rename, resolve collisions, save)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("STEP 5: BUILD MEANS-ONLY FILE")
print("=" * 90)

df_means = df_full[df_full['moment'].isin(MEANS_MOMENTS)].copy()
n_before_rename = len(df_means)

print(f"\n  Filtered to moment in {sorted(MEANS_MOMENTS)}:")
print(f"    {len(df_full):,} -> {len(df_means):,} rows")
print(f"    (dropped: {len(df_full) - len(df_means):,} moment columns "
      f"-- cwstd/cwskew/cwkurt/spread, no means-only equivalent)")

by_moment = df_full['moment'].value_counts()
print(f"\n  Moment breakdown in the full file:")
for m, n in by_moment.items():
    kept = " (kept)" if m in MEANS_MOMENTS else " (dropped from means-only)"
    print(f"    {m:<12s} {n:>5,d}{kept}")

# Rename to the bare base_factor name -- this is what agg_means and panel
# actually use for stock-level columns. This is also the step that surfaces
# stock/macro base_factor name collisions (see KNOWN_COLLISIONS above).
df_means['column'] = df_means['base_factor']

# ── Resolve known collisions before checking for duplicates ─────────────────
dup_names = sorted(
    df_means.loc[df_means['column'].duplicated(keep=False), 'column'].unique()
)

if dup_names:
    print(f"\n  {len(dup_names)} base_factor name collision(s) found after "
          f"renaming to bare names: {dup_names}")

    unresolved = [n for n in dup_names if n not in KNOWN_COLLISIONS]
    assert not unresolved, (
        f"Unresolved column-name collision(s) with no known rule: "
        f"{unresolved}. Inspect these by hand and add a rule to "
        f"KNOWN_COLLISIONS before re-running.")

    for name, keep_level in KNOWN_COLLISIONS.items():
        group = df_means[df_means['column'] == name]
        if len(group) <= 1:
            continue  # this known collision did not actually occur this run

        keep = group[group['level'] == keep_level]
        drop = group[group['level'] != keep_level]
        assert len(keep) == 1, (
            f"Expected exactly one row with level='{keep_level}' for "
            f"'{name}', found {len(keep)} -- collision shape has changed, "
            f"re-investigate before trusting this resolution.")

        print(f"    Collision '{name}': dropping level={drop['level'].tolist()} "
              f"row, keeping level='{keep_level}'")
        df_means = df_means.drop(index=drop.index)
else:
    print(f"\n  ✓ No base_factor name collisions")

n_dupe_means = df_means['column'].duplicated().sum()
assert n_dupe_means == 0, (
    f"{n_dupe_means} duplicate columns remain after resolving known "
    f"collisions -- a new, unhandled collision exists: "
    f"{sorted(df_means.loc[df_means['column'].duplicated(keep=False), 'column'].unique())}")

print(f"\n  {n_before_rename:,} -> {len(df_means):,} rows after collision resolution")

df_means.to_csv(OUT_MEANS_PATH, index=False)
print(f"\n  Saved: {OUT_MEANS_PATH.name}")
print(f"    {len(df_means):,} rows, {len(df_means.columns)} columns")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: FINAL RECONCILIATION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("STEP 6: FINAL RECONCILIATION")
print("=" * 90)

full_subthemes = set(df_full['subtheme_id'])
means_subthemes = set(df_means['subtheme_id'])
print(f"\n  Subthemes in full-moments file:  {len(full_subthemes)}")
print(f"  Subthemes in means-only file:    {len(means_subthemes)}  "
      f"(subset of full: {means_subthemes <= full_subthemes})")

# Means-only IDs are expected to be non-contiguous within a theme (gaps are
# fine), but every ID that DOES appear must trace back to the same theme.
means_theme_check = (
    df_means.drop_duplicates('subtheme_id')
    .merge(df_full.drop_duplicates('subtheme_id')[['subtheme_id', 'theme_id']],
           on='subtheme_id', suffixes=('_means', '_full'))
)
assert (means_theme_check['theme_id_means']
        == means_theme_check['theme_id_full']).all(), (
    "A subtheme_id resolves to different theme_id values between the "
    "full-moments and means-only files -- numbering inheritance is broken.")
print(f"  ✓ Every means-only subtheme_id resolves to the same theme_id as "
      f"in the full-moments file")

# skew_chg_5d specifically: confirm the surviving row in means-only is macro.
if 'skew_chg_5d' in df_means['column'].values:
    row = df_means.loc[df_means['column'] == 'skew_chg_5d',
                        ['column', 'level', 'theme_name', 'subtheme_name', 'subtheme_id']]
    print(f"\n  skew_chg_5d in means-only file (should be exactly one row, level=macro):")
    print(row.to_string(index=False))
    assert len(row) == 1 and row['level'].iloc[0] == 'macro', (
        "skew_chg_5d resolution did not land as expected -- investigate.")

print(f"\n  Per-theme subtheme counts (full-moments file):")
counts = (df_full.drop_duplicates('subtheme_id')
          .groupby(['theme_id', 'theme_name']).size()
          .reset_index(name='n_subthemes').sort_values('theme_id'))
print(counts.to_string(index=False))

print(f"\n  Done.")
print(f"    {OUT_FULL_PATH}")
print(f"    {OUT_MEANS_PATH}")

STAGE 5 THEMES -- 04: NUMBER SUBTHEMES

  Loaded: classified_moment_inventory_long.csv
    1,699 rows
    Themes (raw):     13
    Subthemes (raw):  331
    ✓ No unassigned rows, no duplicate columns

STEP 2: BUILD ID MAPS

  13 themes numbered 01-13
  Largest theme has 65 subthemes

  Theme ID assignment (alphabetical):
    01  Analyst Expectations & Sentiment          41 subthemes   212 features
    02  Credit Conditions                          5 subthemes    24 features
    03  Cross-Sectional Risk Profile              24 subthemes   100 features
    04  Global Markets, FX & Commodities           9 subthemes    51 features
    05  Interest Rates & Monetary Policy          11 subthemes    57 features
    06  Investment & Corporate Structure          28 subthemes   145 features
    07  Liquidity & Market Quality                41 subthemes   197 features
    08  Macroeconomic Fundamentals                16 subthemes    81 features
    09  Momentum & Reversal                       28 

In [1]:
"""
Print a table of every subtheme in the means-only dataset, alongside the
number of means-only factors (columns) belonging to it.

Reads the ALREADY-BUILT output file directly (numbered_classified_moment_
inventory_means_only.csv) rather than recomputing anything -- this is a
read-only reporting script, not part of the numbering pipeline itself.
"""
import pandas as pd
from pathlib import Path

THEMES_DIR = Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes')
MEANS_PATH = THEMES_DIR / 'numbered_classified_moment_inventory_means_only.csv'

# dtype=str on the ID columns as a defensive habit, per the pipeline's own
# stated convention -- underscore IDs can't be misparsed as floats, but
# there's no cost to being explicit.
df_means = pd.read_csv(MEANS_PATH, dtype={'subtheme_id': str, 'theme_id': str})

required = {'column', 'subtheme_id', 'subtheme_name', 'theme_id', 'theme_name'}
missing = required - set(df_means.columns)
assert not missing, f"means-only file missing expected columns: {missing}"

# One row per (subtheme_id, subtheme_name, theme_id, theme_name), with a
# count of how many distinct 'column' values (means-only factors) fall
# under each subtheme.
subtheme_counts = (
    df_means.groupby(['theme_id', 'theme_name', 'subtheme_id', 'subtheme_name'])['column']
            .nunique()
            .reset_index(name='n_means_factors')
            .sort_values('subtheme_id')
            .reset_index(drop=True)
)

n_subthemes = len(subtheme_counts)
n_factors_total = df_means['column'].nunique()

print("=" * 90)
print("MEANS-ONLY DATASET -- SUBTHEMES AND FACTOR COUNTS")
print("=" * 90)
print(f"\n  Total subthemes:         {n_subthemes}")
print(f"  Total means-only factors: {n_factors_total}")
print(f"  (sanity check: sum of per-subtheme counts = "
      f"{subtheme_counts['n_means_factors'].sum()}, "
      f"should equal {n_factors_total} -- every factor belongs to exactly "
      f"one subtheme)")
assert subtheme_counts['n_means_factors'].sum() == n_factors_total, (
    "Per-subtheme counts don't sum to the total factor count -- a factor "
    "is either double-counted or missing a subtheme assignment."
)

print(f"\n{'subtheme_id':<12s}{'theme_id':<10s}{'theme_name':<38s}"
      f"{'subtheme_name':<40s}{'n_means_factors':>16s}")
print("-" * 116)
for _, row in subtheme_counts.iterrows():
    print(f"{row['subtheme_id']:<12s}{row['theme_id']:<10s}"
          f"{row['theme_name'][:36]:<38s}{row['subtheme_name'][:38]:<40s}"
          f"{row['n_means_factors']:>16d}")

# Also save it out as a CSV, since a table this size is easier to skim or
# paste into a write-up as a file than to read off a printed console block.
OUT_PATH = THEMES_DIR / 'means_only_subtheme_factor_counts.csv'
subtheme_counts.to_csv(OUT_PATH, index=False)
print(f"\nSaved: {OUT_PATH}")

MEANS-ONLY DATASET -- SUBTHEMES AND FACTOR COUNTS

  Total subthemes:         128
  Total means-only factors: 574
  (sanity check: sum of per-subtheme counts = 574, should equal 574 -- every factor belongs to exactly one subtheme)

subtheme_id theme_id  theme_name                            subtheme_name                            n_means_factors
--------------------------------------------------------------------------------------------------------------------
01_01       01        Analyst Expectations & Sentiment      Analyst Coverage Breadth                               5
01_05       01        Analyst Expectations & Sentiment      Analyst Forecast Revision                              6
01_09       01        Analyst Expectations & Sentiment      Bearish Positioning Pressure                           4
01_13       01        Analyst Expectations & Sentiment      Consensus Coherence Breakdown                          4
01_17       01        Analyst Expectations & Sentiment      Fundam

In [2]:
"""
Print the per-theme summary table (theme_id, theme_name, n_subthemes,
n_features), but for the MEANS-ONLY dataset -- the same shape of table the
numbering script prints for the full-moments file in Step 2, just computed
from numbered_classified_moment_inventory_means_only.csv instead.

Read-only reporting script; does not touch or regenerate either numbered
CSV.
"""
import pandas as pd
from pathlib import Path

THEMES_DIR = Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes')
MEANS_PATH = THEMES_DIR / 'numbered_classified_moment_inventory_means_only.csv'

df_means = pd.read_csv(MEANS_PATH, dtype={'subtheme_id': str, 'theme_id': str})

required = {'column', 'subtheme_id', 'theme_id', 'theme_name'}
missing = required - set(df_means.columns)
assert not missing, f"means-only file missing expected columns: {missing}"

theme_summary = (
    df_means.groupby(['theme_id', 'theme_name'])
            .agg(
                n_subthemes=('subtheme_id', 'nunique'),
                n_features=('column', 'nunique'),
            )
            .reset_index()
            .sort_values('theme_id')
            .reset_index(drop=True)
)

n_themes = len(theme_summary)
n_subthemes_total = df_means['subtheme_id'].nunique()
n_features_total = df_means['column'].nunique()

print(f"\n  {n_themes} themes (means-only dataset)")
print(f"\n  Theme ID assignment (alphabetical):")
for _, row in theme_summary.iterrows():
    print(f"    {row['theme_id']}  {row['theme_name']:<40s} "
          f"{row['n_subthemes']:>3d} subthemes  {row['n_features']:>4d} features")

# Sanity checks -- sums should reconcile against the totals computed above.
assert theme_summary['n_subthemes'].sum() == n_subthemes_total, (
    "Per-theme subtheme counts don't sum to the total distinct subtheme_id "
    "count -- a subtheme_id is shared across more than one theme, which "
    "should not be possible."
)
assert theme_summary['n_features'].sum() == n_features_total, (
    "Per-theme feature counts don't sum to the total distinct column "
    "count -- a factor is either double-counted across themes or missing "
    "a theme assignment."
)
print(f"\n  Totals: {n_subthemes_total} subthemes, {n_features_total} features "
      f"(reconciled OK)")

OUT_PATH = THEMES_DIR / 'means_only_theme_summary.csv'
theme_summary.to_csv(OUT_PATH, index=False)
print(f"\nSaved: {OUT_PATH}")


  13 themes (means-only dataset)

  Theme ID assignment (alphabetical):
    01  Analyst Expectations & Sentiment          11 subthemes    47 features
    02  Credit Conditions                          5 subthemes    24 features
    03  Cross-Sectional Risk Profile               6 subthemes    20 features
    04  Global Markets, FX & Commodities           9 subthemes    51 features
    05  Interest Rates & Monetary Policy          11 subthemes    57 features
    06  Investment & Corporate Structure           7 subthemes    30 features
    07  Liquidity & Market Quality                11 subthemes    41 features
    08  Macroeconomic Fundamentals                16 subthemes    81 features
    09  Momentum & Reversal                       10 subthemes    40 features
    10  Order Flow & Participation                10 subthemes    48 features
    11  Profitability & Earnings Quality           4 subthemes    18 features
    12  Valuation                                  2 subthemes    18 